# Practical Assignment 9: Sparse Autoencoder to Reconstruct MNIST Digits

**Aim:** Implement a simple autoencoder to reconstruct MNIST digits and add a sparsity constraint on the encoded representations.

**Key Concepts to Remember:**
1. **Autoencoder Principle:** Unsupervised neural network where target = input ($X \to X$).
2. **Sparsity Constraint:** An $L_1$ activity regularization penalty `activity_regularizer=l1(1e-4)` penalizes active neurons, forcing the bottleneck to learn sparse, compact features.
3. **Architecture:** Input (784) $\to$ Encoder (32 sparse) $\to$ Decoder (784 sigmoid).


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Step 1: Load MNIST Data
df = pd.read_csv(r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/MNIST/mnist_784_csv.csv', nrows=6000)
X = df.iloc[:, :-1].values.astype('float32') / 255.0
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
print(f'Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')

# Step 2: Build Sparse Autoencoder Architecture
input_dim = 784
encoding_dim = 32  # Bottleneck layer (compressed representation)

input_img = tf.keras.layers.Input(shape=(input_dim,))
# Encoder with L1 activity regularizer (sparsity constraint)
encoded = tf.keras.layers.Dense(
    encoding_dim, activation='relu',
    activity_regularizer=tf.keras.regularizers.l1(1e-4),
    name='sparse_bottleneck'
)(input_img)
# Decoder reconstructing original 784 pixels
decoded = tf.keras.layers.Dense(input_dim, activation='sigmoid')(encoded)

autoencoder = tf.keras.Model(input_img, decoded)
encoder = tf.keras.Model(input_img, encoded)
autoencoder.summary()

# Step 3: Compile and Train (Input is X, Target is X!)
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')
history = autoencoder.fit(X_train, X_train, epochs=5, batch_size=64, validation_data=(X_test, X_test), verbose=1)

# Step 4: Evaluate Reconstruction and Sparsity
reconstructed = autoencoder.predict(X_test, verbose=0)
mse = np.mean((X_test - reconstructed) ** 2)
print(f'\nTest Reconstruction MSE: {mse:.5f}')

bottleneck_acts = encoder.predict(X_test, verbose=0)
sparsity = np.mean(bottleneck_acts == 0) * 100
print(f'Mean Activation across Bottleneck Neurons: {np.mean(bottleneck_acts):.5f}')
print(f'Sparsity (% of Inactive / Exactly Zero Neurons): {sparsity:.2f}%')


Train samples: 4800, Test samples: 1200
Model: "functional_7"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ input_layer_7 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sparse_bottleneck (Dense)       │ (None, 32)             │        25,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 784)            │        25,872 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 50,992 (199.19 KB)
 Trainable params: 50,992 (199.19 KB)
 Non-trainable params: 0 (0.00 B)
Epoch 1/5

 1/75 ━━━━━━━━━━━━━━━━━━━━ 28s 384ms/step - loss: 0.7334
42/75 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6990   
75/75 ━━━━━━━━